In [ ]:
# Importer les packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import les données
df = pd.read_csv('')

In [ ]:
# Afficher les infos des données
df.info()

### <font color = 'bisque'>Exploration et préparation des données

In [ ]:
# sélection les variables quantitatives
quant_data = df.select_dtypes(include = 'number')
quant_data.head()

In [ ]:
# décrire les variables quantitatives
quant_data.describe()

In [ ]:
# pairplot des variables quantitatives
sns.pairplot(quant_data)
plt.show()

In [ ]:
# Fonction permettant de visualiser les boxplots
def boxplot_func(col):
    fig = px.box(df,x = col)
    fig.show()

In [ ]:
# Appliquer la fonctions boxplot_func
for col in quant_data.columns:
  boxplot_func(col)

In [ ]:
# Afficher la matrice de corrélation
quant_data.corr()

In [ ]:
# Visualisaton le matrice de corrélation
plt.figure(figsize = (20, 10))
sns.heatmap(quant_data.corr(), annot = True)
plt.show()

In [ ]:
# sélection les variables catégorielles
cat_data = df.select_dtypes(include = 'object')

In [ ]:
# Faire un describre
cat_data.describe()

In [ ]:
# Fonction de visualisaion du counplot
def countplot_func(col):
    val = cat_data[col].value_counts()
    fig = px.bar(x = val.index, y = val.values,color = val.index.astype(str), text_auto = True)
    fig.show()

In [ ]:
# Appliquer la fonction countplot_func
for col in cat_data.columns:
  countplot_func(col)

In [ ]:
# Encoder les variables catégorielles
from sklearn.preprocessing import LabelEncoder
# Créer différents encodeurs pour les variables catégorielles
encoder0 = LabelEncoder()
encoder1 = LabelEncoder()
encoder2 = LabelEncoder()
encoders = [encoder0, encoder1, encoder2]
for i in range(len(cat_data.columns)):
  encoders[i].fit(df[cat_data.columns[i]]) # le modèle prend le temps de reconnaitre les classes
  df[cat_data.columns[i]] = encoders[i].transform(df[cat_data.columns[i]]) # encoder les valeurs

In [ ]:
# télécharger les encodeurs en utilisant joblib
import joblib
#
jb.dump(encoders, 'encoders.joblib')

In [ ]:
from sklearn.impute import KNNImputer
# fonction d'imputation des valeurs aberrantes
def impute_func(df, col, n_neighbors=5):
    # 2. Détection des outliers (Méthode IQR)
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    min_value = Q1 - 1.5 * IQR
    max_value = Q3 + 1.5 * IQR

    # Les valeurs hors bornes deviennent des valeurs manquantes pour le KNNImputer
    df.loc[(df[col] < min_value) | (df[col] > max_value), col] = np.nan # NaN
    # n_neighbors : nombre de voisins pour calculer la moyenne de remplacement
    imputer = KNNImputer(n_neighbors=n_neighbors)
    df[:] = imputer.fit_transform(df)

In [ ]:
# Application du traitement
for col in quant_data.columns:
  impute_func(df,col)

In [ ]:
# réafficher les boxplots
for col in quant_data.columns:
  boxplot_func(col)

In [ ]:
# Fractionner les données
x = df.iloc[:,:-1].values
y = df.iloc[:, -1].values

In [ ]:
from sklearn.model_selection import train_test_split
# Splitter les données en train, val et test
x_train, x_vt,y_train,y_vt = train_test_split(x,y , test_size = 0.2, random_state = 42)
# Splitter les données en val et test
x_val, x_test, y_val, y_test = train_test_split(x_vt, y_vt, test_size = 0.5, random_state = 42)

### <font color = 'bisque'>Random Forest Regressor<font>

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestRegressor

# Pipeline
pipeline = Pipeline([
    ('scaler', RobustScaler()),
    ('regressor', RandomForestRegressor())
])
# Dictionnaires des paramètres
param_grid = {
    'regressor__n_estimators':np.arange(10,100,10),
    'regressor__max_depth':np.arange(1,15)
}

# Objet grid
grid = GridSearchCV(pipeline, param_grid, cv=5)
# Lancer la recherche
grid.fit(x_train, y_train)

In [ ]:
# paramètres optimaux


In [ ]:
# Rélancer le pipeline en utilisant les paramètres optimaux
pipe = Pipeline([
    # encodage
    ('scaler', RobustScaler()),
    ('regressor', RandomForestRegressor(max_depth= 14, n_estimators = 70))
])
#
pipe.fit(x_train, y_train)

In [ ]:
# récupérer le pipeline du meilleur modèle(le modèle entrainé avec les paramètres)
pipe_from_grid_ = grid_.best_estimator_

In [ ]:
# Barplot des contributions des variables
def bar_feat_imp(pipe):
  fig= px.bar(x = pipe.named_steps['regressor'].feature_importances_, y = df.columns[:-1])
  fig.show()

In [ ]:
# barplot


#### <font color = 'teal'> Evaluation du Random Forest<font>

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
# fonction de calcul du R2 score, Mean Squarred Error et Mean Absolute Error
def evaluation(pipe):
  y_pred = pipe.predict(x_val)
  # mse
  mse = mean_squared_error(y_val, y_pred)
  # r2
  r2 = r2_score(y_val, y_pred)
  # mae
  mae = mean_absolute_error(y_val, y_pred)
  return mse, r2, mae

In [ ]:
# evaluation
mse_rf, r2_rf, mae_rf = evaluation(pipe_from_grid)
# mse_rf, r2_rf, mae_rf = evaluation(pipe)
print(f"MSE : {mse_rf:.2f} -- R2 : {r2_rf:.2f} -- MAE : {mae_rf:.2f}")

In [ ]:
# Calculer le R2 score sur les données d'entrainement
y_pred_train = pipe_from_grid.predict(x_train)
print(f"R2 sur les données train : {r2_score(y_train, y_pred_train): .2f}")

### <font color = 'bisque'>XGBoost Regressor

In [ ]:
# Rechercher les paramètres optimaux de l'algorithme
from xgboost import XGBRegressor
# Pipeline
pipeline_ = Pipeline([
    ('scaler', RobustScaler()),
    ('regressor', XGBRegressor())
])
# Dictionnaires des paramètres
param_grid_ = {
    'regressor__n_estimators':np.arange(10,100,10),
    'regressor__max_depth':np.arange(1,15)
}

# Objet grid
grid_ = GridSearchCV(pipeline_, param_grid_, cv=5)
# Lancer la recherche
grid_.fit(x_train, y_train)

In [ ]:
# Afficher les paramètres optimaux


In [ ]:
# récupérer le pipeline du meilleur modèle


In [ ]:
# Barplot des contributions des variables


#### <font color = 'teal'> Evaluation du XGBoost Regressor<font>

In [ ]:
# Evaluation


In [ ]:
# Calculer le R2 score sur les données d'entrainement


In [ ]:
# télécharger le modèle le plus performant


### <font color = 'bisque'>Tester le modèle le plus performant<font>

In [ ]:
#  echantillon des données test
x_test_10 = x_test[:10]
y_test_10 = y_test[:10]
# Prédictions
y_pred_10 = xgb_best_model_pipe.predict(x_test_10)
# Parcourir les prédictions
for i in range(10):
  print(f"La voiture {i + 1} est vendu à {y_pred_10[i]:.2f}k$ -- {y_test_10[i]:.2f}k$")